# Task 4 — YOLOv8 5-fold + masked pool (WBF 앙상블용 체크포인트 생성, Colab)

**목적**: task2-masked와 동일한 데이터(원본 train 232장 + 합성 pool2 + **masked pool**, 74종)로 **YOLOv8**을
5-fold 학습해, 추후 RF-DETR(task2-masked)와 fold-matched WBF 앙상블에 쓸 체크포인트 5개를 만든다.
**classification loss 가중치를 높여(cls gain 0.5→1.5)** 앙상블에서 분류 정확도에 기여하는 체크포인트를 목표로 한다.
이 노트북 자체의 산출물은 **체크포인트**이며, Kaggle 제출용 CSV는 만들지 않는다.

| 항목 | 내용 |
|---|---|
| 학습 데이터 | 원본 train 232장(corrections 반영) + 합성 pool2 + **masked pool** — **task2-masked와 완전히 동일** (`syn_00505/00102` 제외 포함) |
| 클래스 | 74종 (train 56종 → 라벨 1~56, test 전용 18종 → 라벨 57~74) |
| fold 분할 | **재계산하지 않음** — task2-masked에서 export한 `fold_split_masked.json`을 Drive에서 로드. 세션/계정/라이브러리 버전 차이로 인한 분할 불일치를 원천 차단 (fold-matched WBF의 전제) |
| 모델/학습 | **YOLOv8m** (YOLO11 대비 가벼움 — masked pool 추가로 데이터가 커져 시간 절약), 100 epochs, imgsz 960, patience 15 |
| loss | box=7.5, **cls=1.5(기본 0.5의 3배)**, dfl=1.5 — Ultralytics 내장 loss gain 인자로 조정 (**loss 코드 수정 없음**) |
| test 추론 | 5-fold 앙상블 → WBF 융합 → 클래스별 crop으로 육안 확인 (제출 CSV 없음) |

**masked pool 처리 (task2-masked와 동일한 유의사항)**
- 파일명이 train과 동일 체계(K코드+촬영조건)라 원본과 겹칠 수 있음 → **전체 파일을 `msk_` 접두어로 리네임**한
  사본을 스테이징 폴더에 만들어 병합 (이미지 캐시 덮어쓰기·corrections 오적용 방지)
- annotation은 **원본 category_id를 그대로 사용** (합성 pool의 라벨 네임스페이스와 다름 — name 매핑 불필요)
- 같은 구성(위도/촬영조건만 다른 이미지 + masked 버전)의 train/valid 누수 방지 그룹화는
  `fold_split_masked.json`에 이미 반영되어 있으며, 로드 후 그룹 누수 assert로 재검증한다
- **masked pool 경로는 자동 탐색하지 않음** → 4번 셀의 `MASKED_ROOT`에 실제 Drive 경로를 직접 기입할 것


**실행 전제**
- Colab: **GPU 런타임** 필요 (Runtime → Change runtime type → GPU)
- Drive에 `fold_split_masked.json` 업로드 완료 상태여야 함 (4번 셀이 파일명으로 재귀 검색)
- `batch`는 실제 배정받은 GPU(T4/L4/A100 등) 메모리에 따라 OOM 시 낮춰야 한다.


In [ ]:
# [1. 환경 준비] ultralytics(YOLOv8 학습/추론 + COCO->YOLO 변환 유틸 포함), ensemble-boxes(WBF)
!pip install -q "ultralytics>=8.3" ensemble-boxes


In [ ]:
# [2. 저장소 준비] 팀 저장소 main 브랜치를 clone하고 RF_DETR_split_ver를 import 경로에 추가합니다.
#  ⚠ 저장소 파일/함수는 수정하지 않고 그대로 재사용합니다. (YOLO 관련 로직은 이 노트북에서 로컬로 정의)
import os, sys, re, glob, json, shutil, math
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
from PIL import Image

REPO_URL = 'https://github.com/kim-tae-yoon-0718/ai12-team01.git'
REPO_DIR = '/content/ai12-team01'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
sys.path.insert(0, os.path.join(REPO_DIR, 'RF_DETR_split_ver'))

# ── 저장소에서 그대로 재사용하는 함수 (경로 탐색 · 데이터 전처리 - 모델과 무관한 순수 로직) ──
from colab_setup import mount_drive
from dataset import (
    find_data_root,       # Drive에서 폴더 이름으로 데이터 루트 자동 탐색
    check_data_paths,     # train/test 이미지·annotation 하위 경로 존재 점검
    load_raw_annotations, # 박스당 1개로 흩어진 원본 annotation을 파일명 기준 병합 로드
    apply_corrections,    # corrections 기반 bbox 좌표/중복/누락/클래스 오기재 수정
    # make_folds는 일부러 import하지 않음 - fold 분할은 fold_split_masked.json 로드로 고정 (재계산 금지)
    cache_images,         # 이미지 로컬 캐시
    write_fold_dirs,      # fold별 {train,valid}에 COCO json + 이미지 배치
    save_label_map,       # label_map.json 저장
    compute_label_counts, # 클래스(라벨)별 박스 수 집계
)

from ultralytics import YOLO
from ultralytics.data.converter import convert_coco   # COCO json -> YOLO txt 공식 변환 유틸
from ensemble_boxes import weighted_boxes_fusion

print('저장소 clone 완료:', REPO_DIR)


In [ ]:
# [3. corrections 하드코딩] task2-masked(Kaggle) 실험과 완전히 동일한 스냅샷을 사용합니다.
#     (fix_category "3444"->3351 및 "791"->31863 포함, add_boxes 1건 category 상이).
#     fold-matched WBF 앙상블 파트너인 task2-masked RF-DETR 체크포인트와 학습 라벨 조건을
#     완전히 일치시키기 위해 의도적으로 이 스냅샷을 고정합니다.
#  apply_corrections()는 "파일 경로"를 받는 시그니처라(저장소 함수 무수정 원칙),
#  dict를 로컬 파일로 1회 저장한 뒤 그 경로를 넘기는 방식으로 우회합니다.
corrections = {
    "coord_fix": {
        "K-003351-016262-018357_0_2_0_2_75_000_200.png": [
            {"original": [6567, 625, 311, 315], "corrected": [567, 625, 311, 315]}
        ]
    },
    "remove_boxes": {
        "K-001900-016548-019607-033009_0_2_0_2_70_000_200.png": [
            {"category_id": 16548, "bbox": [88, 255, 366, 209]}
        ]
    },
    "modify_boxes": {
        "K-003351-020014-020238_0_2_0_2_90_000_200.png": [
            {"category_id": 3351, "match_bbox": None, "new_bbox": [390, 260, 170, 165]}
        ],
        "K-003351-019232-029667_0_2_1_2_70_000_200.png": [
            {"category_id": 19232, "match_bbox": None, "directive": "EXTEND_DOWN_95"}
        ]
    },
    "add_boxes": {
        "K-001900-016548-019607-033009_0_2_0_2_70_000_200.png": [
            {"category_id": 16548, "bbox": [90, 870, 245, 240]}
        ],
        "K-003351-013900-021325_0_2_0_2_70_000_200.png": [
            {"category_id": 3351, "bbox": [400, 830, 180, 180]}
        ],
        "K-003351-013900-036637_0_2_0_2_70_000_200.png": [
            {"category_id": 3351, "bbox": [440, 880, 175, 175]}
        ],
        "K-003351-020014-022074_0_2_0_2_90_000_200.png": [
            {"category_id": 20014, "bbox": [65, 720, 325, 315]}
        ],
        "K-003351-020238-031863_0_2_0_2_70_000_200.png": [
            {"category_id": 20238, "bbox": [580, 290, 215, 215]}
        ],
        "K-003351-021325-032310_0_2_0_2_90_000_200.png": [
            {"category_id": 32310, "bbox": [595, 830, 345, 245]}
        ],
        "K-003351-029667-031863_0_2_0_2_70_000_200.png": [
            {"category_id": 3351, "bbox": [375, 870, 165, 165]}
        ],
        "K-003351-032310-038162_0_2_0_2_70_000_200.png": [
            {"category_id": 3351, "bbox": [390, 855, 185, 185]}
        ],
        "K-003351-033880-038162_0_2_0_2_75_000_200.png": [
            {"category_id": 33880, "bbox": [70, 600, 310, 425]}
        ],
        "K-003351-035206-041768_0_2_0_2_70_000_200.png": [
            {"category_id": 3351, "bbox": [460, 875, 180, 180]}
        ],
        "K-003544-004543-012247-016548_0_2_0_2_90_000_200.png": [
            {"category_id": 4543, "bbox": [640, 195, 205, 190]}
        ]
    },
    "fix_category": {
        "791": 31863,
        "3444": 3351,
        "3441": 3351,
        "1420": 35206,
        "1412": 27733
    }
}

CORRECTIONS_PATH = '/content/corrections.json'
with open(CORRECTIONS_PATH, 'w', encoding='utf-8') as f:
    json.dump(corrections, f, ensure_ascii=False, indent=2)
print('corrections 저장 (task2-masked 동일 스냅샷):', CORRECTIONS_PATH)


In [ ]:
# [4. Drive 마운트 + 경로 자동 탐색]
#  원본 train과 pool2가 모두 팀 공유 Drive 폴더 하위에 있다는 전제로, 폴더 "이름"으로 찾습니다.
#  팀원마다 상위 폴더 구조가 달라도(예: '1팀 공유 문서' 하위 여부) 동작하도록 후보 경로를 우선 탐색하고,
#  실패하면 search_root 아래를 재귀 검색합니다 (dataset.find_data_root 그대로 재사용).
mount_drive()

SEARCH_ROOT = '/content/drive/MyDrive'

DATA_ROOT = find_data_root(
    candidates=[
        '/content/drive/MyDrive',
        '/content/drive/MyDrive',
        '/content/drive/MyDrive',
    ],
    search_root=SEARCH_ROOT,
    target_name='sprint_ai_project1_data',
)
check_data_paths(DATA_ROOT)

POOL_ROOT = find_data_root(
    candidates=[
        '/content/drive/MyDrive/task2_synthesized',
        '/content/drive/MyDrive/task2_synthesized',
    ],
    search_root=SEARCH_ROOT,
    target_name='task2_synthesized',
)
POOL_IMG_DIR = os.path.join(POOL_ROOT, 'images')
POOL_ANN_PATH = (glob.glob(os.path.join(POOL_ROOT, 'annotations', '*.json')) or [None])[0]

print('DATA_ROOT:', DATA_ROOT)
print('POOL_ROOT:', POOL_ROOT)
print('POOL_IMG_DIR 존재:', os.path.exists(POOL_IMG_DIR) if POOL_IMG_DIR else False)
print('POOL_ANN_PATH:', POOL_ANN_PATH)
assert POOL_ANN_PATH, 'pool2 annotation json을 못 찾음 - 폴더 구조 확인 필요 (images/, annotations/*.json)'

# ── masked pool: 다른 위치에 있어 자동 탐색하지 않습니다. 실제 Drive 경로를 직접 기입하세요. ──
#    (폴더 안에 images/ 와 annotations/_annotations.coco.json 이 있어야 합니다)
MASKED_ROOT = '/content/drive/MyDrive/*** 여기에 masked pool(dataset-74-masked) 경로 기입 ***'   # TODO: 직접 기입
MASKED_IMG_DIR = os.path.join(MASKED_ROOT, 'images')
MASKED_ANN_PATH = (glob.glob(os.path.join(MASKED_ROOT, 'annotations', '*.json')) or [None])[0]
assert os.path.isdir(MASKED_IMG_DIR), (
    f'masked pool images 폴더 없음: {MASKED_IMG_DIR}\n-> 위 MASKED_ROOT를 실제 경로로 수정하세요')
assert MASKED_ANN_PATH, f"masked pool annotation json 없음: {os.path.join(MASKED_ROOT, 'annotations')}"

# ── 고정 fold 분할: task2-masked에서 export한 fold_split_masked.json을 Drive에서 파일명으로 검색 ──
_hits = glob.glob(os.path.join(SEARCH_ROOT, '**', 'fold_split_masked.json'), recursive=True)
assert _hits, f'fold_split_masked.json을 {SEARCH_ROOT} 아래에서 못 찾음 - 업로드 위치/파일명 확인'
if len(_hits) > 1:
    print(f'fold_split_masked.json 후보 {len(_hits)}개 -> 첫 번째 사용:\n  ' + '\n  '.join(_hits))
FOLD_SPLIT_JSON = _hits[0]

print('MASKED_ROOT:', MASKED_ROOT)
print('MASKED_ANN_PATH:', MASKED_ANN_PATH)
print('FOLD_SPLIT_JSON:', FOLD_SPLIT_JSON)


In [ ]:
# [5. 실험 설정]
#  task2-masked(RF-DETR medium, 100ep)와 동일한 데이터 조건에서 YOLOv8m을 학습합니다.
#  imgsz=960 유지: 원본 976x1280의 알약 디테일 보존 (32의 배수).
#  loss gain — Ultralytics 내장 하이퍼파라미터로 loss 성분별 가중치를 조절합니다 (loss 코드 무수정).
#   total = box_gain*box_loss + cls_gain*cls_loss + dfl_gain*dfl_loss  (v8DetectionLoss)
#   기본값 box=7.5, cls=0.5, dfl=1.5 (box 위주) -> cls를 0.5에서 1.5(3배)로 올려 분류 오류에
#   더 큰 패널티를 부여. WBF 앙상블에서 classification 정확도에 기여하는 체크포인트가 목적.
#   ⚠ 학습 후 results.png에서 box_loss 수렴이 눈에 띄게 나빠졌는지 확인할 것 (나빠졌으면 cls 하향 재조정).
TASK_ID = 4
TAG = 'yolov8m_task4_syn74_masked'

config = {
    'data': {
        'n_splits': 5,
        'seed': 42,          # (참고) fold 분할은 fold_split_masked.json 로드로 고정 - seed는 분할에 관여하지 않음
        'dataset_dir': '/content/dataset',             # COCO 포맷 fold 디렉토리 (write_fold_dirs 산출물)
        'cache_dir': '/content/img_cache',
        'yolo_dataset_dir': '/content/yolo_dataset',   # YOLO 포맷(images/labels) fold 디렉토리
    },
    'model': {
        'variant': 'medium',   # nano | small | medium | large | xlarge (yolov8 계열)
        'tag': TAG,
    },
    'train': {
        'epochs': 50,
        'imgsz': 960,
        'batch': 8,             # Colab GPU 메모리에 따라 조정 (OOM 시 4로 축소)
        'patience': 10,         # val 개선이 15ep 없으면 조기 종료 (데이터 증가에 따른 시간 절약)
        'seed': 42,
        'box_gain': 7.5,        # 기본값 유지
        'cls_gain': 1.5,        # 기본 0.5의 3배 - classification loss 패널티 강화
        'dfl_gain': 1.5,        # 기본값 유지
    },
    'output': {
        'local_output_dir': '/content/outputs',
        # 체크포인트/학습곡선을 Drive에 영구 저장 - 세션이 끊겨도 재마운트만 하면 이어하기 가능
        'backup_dir': '/content/drive/MyDrive',
    },
}

for d in (config['data']['cache_dir'], config['data']['dataset_dir'],
          config['data']['yolo_dataset_dir'], config['output']['local_output_dir'],
          config['output']['backup_dir']):
    os.makedirs(d, exist_ok=True)

VIS_SCORE_THR = 0.3   # test 예측 시각화용 confidence threshold (QA 목적 - 제출용 아님)
WBF_IOU_THR = 0.55

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU(!) - 런타임 유형 확인 필요')
print('backup_dir(영구 저장 - Drive):', config['output']['backup_dir'])
print('loss gains:', {k: v for k, v in config['train'].items() if k.endswith('_gain')})


In [ ]:
# [6. 원본 train 로드 + annotation 수정] Task2와 완전히 동일한 절차입니다.
boxes_by_image, cats_by_image, img_meta, ids_by_image = load_raw_annotations(
    os.path.join(DATA_ROOT, 'train_annotations'))
print(f"수정 전: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")

boxes_by_image, cats_by_image = apply_corrections(
    boxes_by_image, cats_by_image, ids_by_image, CORRECTIONS_PATH)
print(f"수정 후: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")

train_cats = sorted({c for cs in cats_by_image.values() for c in cs})
print('train 클래스 수:', len(train_cats))


In [ ]:
# [7. pool2 병합] Task2(Kaggle) 노트북과 완전히 동일한 로직입니다.
#  pool의 _annotations.coco.json은 "라벨 네임스페이스"(id 1~74, name=원본 category_id)로 작성되어 있어,
#  categories의 name 필드로 원본 id 공간으로 되돌린 뒤 병합합니다.
#  ⚠ fold 구성을 Task2와 일치시켜야 하므로, 이 병합 로직은 Task2 노트북과 토씨 하나 다르지 않게 유지합니다.
def load_pool_annotations(pool_ann_path):
    """합성 pool COCO json -> (boxes, cats(원본 id 공간), img_meta, 원본 coco dict)"""
    with open(pool_ann_path, encoding='utf-8') as f:
        coco = json.load(f)
    label2cat_pool = {c['id']: int(c['name']) for c in coco['categories'] if c['id'] != 0}
    fn_by_img_id = {im['id']: im['file_name'] for im in coco['images']}
    p_boxes, p_cats, p_meta = defaultdict(list), defaultdict(list), {}
    for im in coco['images']:
        p_meta[im['file_name']] = (im['width'], im['height'])
    for a in coco['annotations']:
        fn = fn_by_img_id[a['image_id']]
        p_boxes[fn].append([float(v) for v in a['bbox']])
        p_cats[fn].append(label2cat_pool[a['category_id']])
    return p_boxes, p_cats, p_meta, coco

pool_boxes, pool_cats, pool_meta, pool_coco = load_pool_annotations(POOL_ANN_PATH)
print(f"pool2: 이미지 {len(pool_meta)}장 / 박스 {sum(len(v) for v in pool_boxes.values())}개"
      f" / 클래스 {len({c for cs in pool_cats.values() for c in cs})}종")

overlap = set(pool_meta) & set(boxes_by_image)
assert not overlap, f'train/pool 파일명 충돌: {sorted(overlap)[:5]}'

empty_pool = [fn for fn in pool_meta if not pool_boxes.get(fn)]
if empty_pool:
    print(f'박스 0개인 pool 이미지 {len(empty_pool)}장 제외:', empty_pool[:5])

for fn in pool_meta:
    if pool_boxes.get(fn):
        boxes_by_image[fn] = pool_boxes[fn]
        cats_by_image[fn] = pool_cats[fn]
        img_meta[fn] = pool_meta[fn]

print(f"병합 후: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")


In [ ]:
# [7-1. masked pool 병합] dataset-74-masked (실사 기반 masked pool)
#  task2-masked(Kaggle) 노트북과 동일한 로직입니다. 요점:
#   - masked pool의 _annotations.coco.json은 categories id가 "원본 category_id 그대로"라 name 매핑 없이 사용
#   - 파일명이 train과 동일 체계(K코드+촬영조건)라 원본과 겹칠 수 있어, 전체 파일을 'msk_' 접두어로
#     리네임한 사본을 스테이징 폴더에 생성 (cache_images 덮어쓰기·corrections 오적용 방지)
#   - 같은 구성(위도/촬영조건 변형 + masked 버전)의 split 누수 방지 그룹화는 fold_split_masked.json에
#     이미 반영되어 있고, 9번 셀에서 그룹 누수 assert로 재검증합니다
MASKED_STAGE_DIR = '/content/masked_renamed'
os.makedirs(MASKED_STAGE_DIR, exist_ok=True)

def load_masked_annotations(ann_path):
    """masked pool COCO json -> (boxes, cats, meta, coco). category_id를 원본 id로 그대로 사용합니다."""
    with open(ann_path, encoding='utf-8') as f:
        coco = json.load(f)
    fn_by_img_id = {im['id']: im['file_name'] for im in coco['images']}
    m_boxes, m_cats, m_meta = defaultdict(list), defaultdict(list), {}
    for im in coco['images']:
        m_meta[im['file_name']] = (im['width'], im['height'])
    for a in coco['annotations']:
        fn = fn_by_img_id[a['image_id']]
        m_boxes[fn].append([float(v) for v in a['bbox']])
        m_cats[fn].append(int(a['category_id']))
    return m_boxes, m_cats, m_meta, coco

masked_boxes, masked_cats, masked_meta, masked_coco = load_masked_annotations(MASKED_ANN_PATH)
print(f"masked pool: 이미지 {len(masked_meta)}장 / 박스 {sum(len(v) for v in masked_boxes.values())}개"
      f" / 클래스 {len({c for cs in masked_cats.values() for c in cs})}종")

# 안전장치 1: masked pool의 category_id가 74종 매핑(원본 id 공간 = pool2 JSON의 name)에 전부 포함되는지 확인
cats_74 = {int(c['name']) for c in pool_coco['categories'] if c['id'] != 0}
unknown = sorted({c for cs in masked_cats.values() for c in cs} - cats_74)
assert not unknown, f'74종 매핑에 없는 masked pool category_id: {unknown} (원본 id/라벨 공간 착오 확인)'

# 안전장치 2: annotation에는 있는데 실제 이미지 파일이 없는 항목 확인
masked_img_src = {os.path.basename(p): p
                  for p in glob.glob(os.path.join(MASKED_IMG_DIR, '**', '*.png'), recursive=True)}
missing_img = sorted(set(masked_meta) - set(masked_img_src))
assert not missing_img, f'annotation에는 있는데 이미지가 없는 파일: {missing_img[:5]}'

# 안전장치 3: 박스 0개 이미지는 제외 (task2-masked와 동일)
empty_masked = [fn for fn in masked_meta if not masked_boxes.get(fn)]
if empty_masked:
    print(f'박스 0개인 masked 이미지 {len(empty_masked)}장 제외:', empty_masked[:5])

# 리네임('msk_' + 원본 파일명) + 스테이징 복사(Drive -> 로컬) + annotation 병합
n_merged = 0
for fn in masked_meta:
    if not masked_boxes.get(fn):
        continue
    new_fn = 'msk_' + fn
    assert new_fn not in boxes_by_image, f'리네임 후에도 파일명 충돌: {new_fn}'
    dst = os.path.join(MASKED_STAGE_DIR, new_fn)
    if not os.path.exists(dst):
        shutil.copy(masked_img_src[fn], dst)
    boxes_by_image[new_fn] = masked_boxes[fn]
    cats_by_image[new_fn] = masked_cats[fn]
    img_meta[new_fn] = masked_meta[fn]
    n_merged += 1

print(f'masked pool {n_merged}장 리네임 병합 완료 (스테이징: {MASKED_STAGE_DIR})')
print(f"병합 후: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")


In [ ]:
# [7-2. 데이터 제외] task2-masked와 동일한 제외 목록을 반드시 그대로 적용합니다.
#  ⚠ fold_split_masked.json은 이 제외가 적용된 파일 집합 기준으로 만들어졌으므로,
#     목록이 다르면 9번 셀의 "분할-데이터 일치" assert에서 중단됩니다.
EXCLUDE_FILES = [
    'syn_00505.png',
    'syn_00102.png',
]

for fn in EXCLUDE_FILES:
    if fn in boxes_by_image:
        for d in (boxes_by_image, cats_by_image, img_meta):
            d.pop(fn, None)
        print('이미지 제외:', fn)
    else:
        print('⚠ 목록에 없는 파일명(오타 확인):', fn)

print(f"제외 처리 후: 이미지 {len(boxes_by_image)}장 / 박스 {sum(len(v) for v in boxes_by_image.values())}개"
      f" / 클래스 {len({c for cs in cats_by_image.values() for c in cs})}종")


In [ ]:
# [8. 74종 라벨 매핑] Task2와 동일한 체계: train 56종 -> 1~56, test 전용 18종 -> 57~74.
#  fold 구성을 Task2와 맞추려면 이 매핑도 완전히 동일해야 합니다 (make_folds의 층화 라벨 계산에 사용됨).
cat2label = {int(c['name']): c['id'] for c in pool_coco['categories'] if c['id'] != 0}
label2cat = {v: k for k, v in cat2label.items()}
all_cats = [label2cat[l] for l in sorted(label2cat)]

for i, c in enumerate(sorted(train_cats), start=1):
    assert cat2label.get(c) == i, f'train 클래스 {c}가 라벨 {cat2label.get(c)}에 매핑됨 (기대: {i})'

merged_cats = {c for cs in cats_by_image.values() for c in cs}
missing = sorted(merged_cats - set(cat2label))
assert not missing, f'매핑에 없는 클래스 존재: {missing}'

test_only_labels = sorted(set(cat2label.values()) - set(range(1, len(train_cats) + 1)))
print(f'전체 {len(cat2label)}종 매핑 | train {len(train_cats)}종 -> 1~{len(train_cats)}'
      f' | test 전용 라벨: {test_only_labels}')


In [ ]:
# [9. 5-fold 분할 로드 + fold별 클래스 배분 점검] - 고정 분할 (재계산 없음)
#  StratifiedGroupKFold를 다시 돌리지 않고, task2-masked에서 export한 fold_split_masked.json을
#  그대로 로드합니다. 이유:
#   1) fold-matched WBF: task2-masked RF-DETR 체크포인트와 fold별 valid 이미지 집합이 동일해야 함
#   2) 분할 재계산은 세션/계정 간 sklearn·numpy 버전 차이로 다른 분할이 나올 위험이 있음
with open(FOLD_SPLIT_JSON, encoding='utf-8') as f:
    fixed_split = json.load(f)

file_names = sorted(boxes_by_image)
name_to_idx = {fn: i for i, fn in enumerate(file_names)}

folds = []
for fi in range(config['data']['n_splits']):
    tr_names = fixed_split[f'fold{fi}']['train']
    va_names = fixed_split[f'fold{fi}']['valid']
    # 안전장치: 저장된 분할이 이번 세션의 병합 데이터와 완전히 일치하는지 확인
    #  (masked/pool2 파일 목록·제외 목록이 export 시점과 다르면 여기서 바로 에러로 잡힘)
    only_in_split = (set(tr_names) | set(va_names)) - set(file_names)
    only_in_data = set(file_names) - (set(tr_names) | set(va_names))
    assert not only_in_split and not only_in_data, (
        f'fold{fi} 분할-데이터 불일치\n'
        f'  분할에만 있는 파일 예: {sorted(only_in_split)[:5]}\n'
        f'  데이터에만 있는 파일 예: {sorted(only_in_data)[:5]}')
    folds.append((np.array([name_to_idx[fn] for fn in tr_names]),
                  np.array([name_to_idx[fn] for fn in va_names])))
print('고정 fold 분할 로드 완료 (재계산 없음):', FOLD_SPLIT_JSON)

# 그룹 누수 점검: 같은 그룹('msk_' 접두어 제거 기준 구성 코드)이 train/valid 양쪽에 있으면 안 됩니다.
_g = lambda fn: (fn[len('msk_'):] if fn.startswith('msk_') else fn).split('_0_2')[0]
for fi, (tr, va) in enumerate(folds):
    leak = {_g(file_names[i]) for i in tr} & {_g(file_names[i]) for i in va}
    assert not leak, f'fold {fi} 그룹 누수: {sorted(leak)[:5]}'
print('그룹 누수 없음 (masked/원본 동일 구성은 항상 같은 fold 쪽)')

def summarize_fold_distribution(folds, file_names, cats_by_image, cat2label):
    """fold별 train/valid 이미지·박스 수와 클래스 커버리지를 점검합니다. (로컬 정의 - 저장소에 없음)"""
    all_labels = set(cat2label.values())
    rows, val_pivot = [], {}
    for fi, (tr, va) in enumerate(folds):
        def label_box_counts(idxs):
            cnt = defaultdict(int)
            for i in idxs:
                for c in cats_by_image[file_names[i]]:
                    cnt[cat2label[c]] += 1
            return cnt
        tr_cnt, va_cnt = label_box_counts(tr), label_box_counts(va)
        rows.append({
            'fold': fi,
            'train_imgs': len(tr), 'valid_imgs': len(va),
            'train_boxes': sum(tr_cnt.values()), 'valid_boxes': sum(va_cnt.values()),
            'train_missing_labels': sorted(all_labels - set(tr_cnt)),
            'valid_missing_labels': sorted(all_labels - set(va_cnt)),
        })
        val_pivot[f'fold{fi}_valid'] = va_cnt
    summary = pd.DataFrame(rows)
    valid_pivot = pd.DataFrame(val_pivot).reindex(sorted(all_labels)).fillna(0).astype(int)
    valid_pivot.index.name = 'label'
    return summary, valid_pivot

fold_summary, valid_pivot = summarize_fold_distribution(folds, file_names, cats_by_image, cat2label)
display(fold_summary)

for _, r in fold_summary.iterrows():
    if r['train_missing_labels']:
        print(f"⚠ fold {r['fold']}: train에 없는 라벨 {r['train_missing_labels']}")
    if r['valid_missing_labels']:
        print(f"(참고) fold {r['fold']}: valid에 없는 라벨 {r['valid_missing_labels']}")

valid_pivot   # 라벨 x fold별 valid 박스 수 (셀 마지막 줄 -> 표로 표시. 12번 셀 클래스별 집계에서 재사용)


In [ ]:
# 이 셀 바로 위에 추가해서 확인
print("=== category mapping 확인 ===")
print(f"총 클래스 수: {len(cat2label)}")
print(f"\n{'category_id':>12} {'dl_idx':>10}")
print("-" * 25)
for name, cat_id in sorted(cat2label.items(), key=lambda x: x[1]):
    print(f"  {cat_id:>10} {name:>10}")

In [ ]:
# [10. COCO 포맷 fold 디렉토리 생성] Task2와 동일한 저장소 함수 조합입니다.
#  이 산출물(fold{i}/{train,valid}/_annotations.coco.json + 이미지)을 다음 셀에서 YOLO 포맷으로 변환합니다.
cache_images(os.path.join(DATA_ROOT, 'train_images'), config['data']['cache_dir'])
cache_images(POOL_IMG_DIR, config['data']['cache_dir'])
cache_images(MASKED_STAGE_DIR, config['data']['cache_dir'])   # masked 리네임 사본(msk_*.png)도 같은 캐시에 추가

write_fold_dirs(folds, file_names, boxes_by_image, cats_by_image, img_meta,
                cat2label, all_cats, config['data']['cache_dir'], config['data']['dataset_dir'])

# fold 디렉토리 복사가 끝나면 캐시는 더 필요 없으므로 삭제해 디스크를 확보합니다
#  (masked pool 추가로 데이터가 커져 fold 5개 사본 + 캐시가 공존하면 디스크가 부족할 수 있음)
shutil.rmtree(config['data']['cache_dir'], ignore_errors=True)
print('이미지 캐시 삭제 완료 (fold 디렉토리에 이미 복사됨)')

save_label_map(cat2label, label2cat, config['data']['dataset_dir'])
print('COCO 포맷 fold 디렉토리 생성 완료:', config['data']['dataset_dir'])


## 11. YOLO 포맷 변환

Ultralytics는 객체 검출 학습에 COCO json을 직접 쓰지 않고 `images/` + `labels/`(YOLO txt) 구조와
`data.yaml`을 요구한다 (확인 결과: 세그멘테이션/포즈가 아닌 detection 학습에 COCO json을 그대로 읽는
네이티브 경로는 없음). 다만 공식 변환 유틸 `ultralytics.data.converter.convert_coco()`가 이 변환을
대신해주므로 직접 파싱하지 않고 그대로 사용한다.

- `cls91to80=False`로 호출하면 `class_index = category_id - 1`을 그대로 쓴다 (COCO 80/91클래스
  리매핑을 켜지 않음 — 우리 데이터는 원래 COCO 80종이 아니므로 반드시 꺼야 함).
- 저장소 `build_coco()`가 넣는 더미 배경 카테고리(`id=0`, name='pill')에는 박스가 하나도 없으므로,
  별도로 제거하지 않아도 실제 74종이 정확히 YOLO class index `0~73`으로 매핑된다.
- `convert_coco()`는 라벨 txt만 생성하고 이미지는 복사하지 않으므로, 이미지는 심볼릭 링크로 연결해
  디스크 중복을 피한다.
- 변환된 txt 파일명은 COCO json의 `file_name`과 동일한 stem을 쓰므로, `images/`와 `labels/`를
  같은 파일명 규칙으로 나란히 두면 Ultralytics가 자동으로 짝을 찾는다.


In [ ]:
# [11. YOLO 포맷 변환 실행]
def build_yolo_fold(fold_idx, coco_dataset_dir, yolo_dataset_dir):
    """fold{fold_idx}의 COCO 포맷(train/valid)을 YOLO 포맷(images/labels)으로 변환합니다.

    Args:
        fold_idx (int): fold 번호
        coco_dataset_dir (str): write_fold_dirs()의 output_dir (COCO 포맷 fold 루트)
        yolo_dataset_dir (str): YOLO 포맷 fold를 생성할 루트

    Returns:
        str: 이 fold의 data.yaml 경로
    """
    fold_root = os.path.join(yolo_dataset_dir, f'fold{fold_idx}')

    for split in ('train', 'valid'):
        coco_split_dir = os.path.join(coco_dataset_dir, f'fold{fold_idx}', split)
        img_dst = os.path.join(fold_root, split, 'images')
        lbl_dst = os.path.join(fold_root, split, 'labels')
        os.makedirs(img_dst, exist_ok=True)
        os.makedirs(lbl_dst, exist_ok=True)

        # 1) 이미지: 복사 대신 심볼릭 링크 (COCO 포맷 fold 디렉토리와 디스크 중복 방지)
        for src in glob.glob(os.path.join(coco_split_dir, '*.png')):
            link = os.path.join(img_dst, os.path.basename(src))
            if not os.path.exists(link):
                os.symlink(os.path.abspath(src), link)

        # 2) 라벨: convert_coco()로 COCO json -> YOLO txt 변환 후 이동
        #    labels_dir은 *.json을 비재귀 탐색하므로, json 1개만 있는 split 폴더를 그대로 넘기면 됩니다.
        tmp_convert_dir = os.path.join(yolo_dataset_dir, '_convert_tmp', f'fold{fold_idx}_{split}')
        shutil.rmtree(tmp_convert_dir, ignore_errors=True)
        convert_coco(labels_dir=coco_split_dir, save_dir=tmp_convert_dir,
                    use_segments=False, use_keypoints=False, cls91to80=False)

        for txt_path in glob.glob(os.path.join(tmp_convert_dir, 'labels', '*', '*.txt')):
            shutil.move(txt_path, os.path.join(lbl_dst, os.path.basename(txt_path)))
        shutil.rmtree(tmp_convert_dir, ignore_errors=True)

        n_imgs = len(glob.glob(os.path.join(img_dst, '*.png')))
        n_lbls = len(glob.glob(os.path.join(lbl_dst, '*.txt')))
        print(f'fold{fold_idx}/{split}: 이미지 {n_imgs} / 라벨 {n_lbls}')

    # 3) data.yaml: names는 YOLO class index(0-based) -> 원본 category_id 문자열
    #    (class_index = category_id - 1 이므로, label(1~74)과 index+1이 정확히 대응)
    names = {i: str(label2cat[i + 1]) for i in range(len(label2cat))}
    yaml_path = os.path.join(fold_root, 'data.yaml')
    with open(yaml_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump({
            'path': os.path.abspath(fold_root),
            'train': 'train/images',
            'val': 'valid/images',
            'names': names,
        }, f, allow_unicode=True, sort_keys=False)
    return yaml_path

fold_yaml_paths = [
    build_yolo_fold(fi, config['data']['dataset_dir'], config['data']['yolo_dataset_dir'])
    for fi in range(config['data']['n_splits'])
]
print('data.yaml 경로:')
for p in fold_yaml_paths:
    print(' ', p)


In [ ]:
# [12. YOLOv8 모델 헬퍼] 저장소 model.py의 RF-DETR variant 매핑과 같은 패턴입니다 (라이브러리가 달라 별도 정의).
#  ultralytics 패키지(8.x)는 yolov8*/yolo11* 가중치를 모두 지원하므로 파일명만 v8 계열로 바꾸면 됩니다.
_YOLO_VARIANTS = {
    'nano': 'yolov8n.pt', 'small': 'yolov8s.pt', 'medium': 'yolov8m.pt',
    'large': 'yolov8l.pt', 'xlarge': 'yolov8x.pt',
}

def get_yolo_model(variant='medium', checkpoint_path=None):
    """YOLOv8 모델을 생성합니다. checkpoint_path가 주어지면 그 가중치로 로드합니다."""
    weights = checkpoint_path or _YOLO_VARIANTS.get(variant.lower())
    if weights is None:
        raise ValueError(f"알 수 없는 YOLOv8 variant: {variant} (지원: {list(_YOLO_VARIANTS)})")
    return YOLO(weights)


In [ ]:
# [13. fold 학습 + 리포팅] 저장소 train.py의 train_fold()/report_fold_result() 패턴을 YOLO용으로 재정의합니다.
#  - backup_dir(Drive)에 이미 {tag}_fold{i}_best.pt가 있으면 학습을 건너뜁니다 (이어하기).
#  - loss 가중치는 Ultralytics 내장 train 인자(box/cls/dfl)로 전달합니다. v8DetectionLoss가
#    box_gain*box + cls_gain*cls + dfl_gain*dfl 가중합을 계산하므로 loss 코드 수정이 필요 없습니다.
#  - Ultralytics는 학습 종료 시 results.csv/results.png(학습 곡선)를 자체 생성하므로 그대로 백업합니다.
def train_fold_yolo(fold_idx, fold_yaml, model_variant, model_tag, train_cfg,
                    local_output_dir, backup_dir):
    """fold 하나를 학습하고 best 체크포인트를 backup_dir에 복사합니다. 백업이 이미 있으면 건너뜁니다."""
    exp = f'{model_tag}_fold{fold_idx}'
    dst = os.path.join(backup_dir, f'{exp}_best.pt')

    if os.path.exists(dst):
        print(f'[fold {fold_idx}] 백업 존재 → 건너뜀')
        return dst

    print(f"\n{'='*50}\n[fold {fold_idx}] 학습 시작\n{'='*50}")
    model = get_yolo_model(model_variant)
    model.train(
        data=fold_yaml,
        epochs=train_cfg['epochs'],
        imgsz=train_cfg['imgsz'],
        batch=train_cfg['batch'],
        patience=train_cfg['patience'],
        seed=train_cfg['seed'],
        box=train_cfg['box_gain'],
        cls=train_cfg['cls_gain'],   # 기본 0.5 -> 1.5: 분류 오류 패널티 강화 (WBF 앙상블에서 분류 담당)
        dfl=train_cfg['dfl_gain'],
        project=local_output_dir,
        name=exp,
        exist_ok=True,
        plots=True,
    )

    os.makedirs(backup_dir, exist_ok=True)
    run_dir = os.path.join(local_output_dir, exp)
    src = os.path.join(run_dir, 'weights', 'best.pt')
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'[fold {fold_idx}] best 백업 → {dst}')
    else:
        dst = None
        print(f'[fold {fold_idx}] best.pt 없음 — 백업 실패')

    for fn in ('results.csv', 'results.png'):
        p = os.path.join(run_dir, fn)
        if os.path.exists(p):
            shutil.copy(p, os.path.join(backup_dir, f'{exp}_{fn}'))

    del model
    torch.cuda.empty_cache()
    return dst


def report_fold_result_yolo(fold_idx, checkpoint_path, fold_yaml):
    """fold 하나의 valid mAP를 계산합니다. 학습을 건너뛴 fold도 체크포인트를 다시 로드해 평가합니다.

    Returns:
        dict: {'map'(mAP@0.5:0.95), 'map_50', 'map_per_class'(class index 0~nc-1 정렬 배열), 'names'}
    """
    model = get_yolo_model(checkpoint_path=checkpoint_path)
    metrics = model.val(data=fold_yaml, split='val', plots=False, verbose=False)
    result = {
        'map': float(metrics.box.map),
        'map_50': float(metrics.box.map50),
        'map_per_class': np.asarray(metrics.box.maps, dtype=float),
        'names': metrics.names,
    }
    del model
    torch.cuda.empty_cache()
    return result


## 14. 5-fold 학습 실행

`run_kfold_yolo()` 하나로 fold마다 다음이 자동 수행됩니다.

1. `train_fold_yolo()` — 학습 후 best 체크포인트를 Drive `backup_dir`에 백업, `results.csv`/`results.png`도 함께 백업
2. `report_fold_result_yolo()` — 체크포인트를 다시 로드해 valid mAP 계산

**이어하기 (Colab 전용 장점)**: `backup_dir`가 Drive 경로이므로, 세션이 끊겨도 **런타임을 다시 시작하고
위 셀들을 순서대로 재실행한 뒤 이 셀만 다시 실행하면** 이미 완료된 fold는 자동으로 스킵됩니다
(Kaggle처럼 체크포인트를 Dataset으로 재업로드할 필요가 없음). fold 구성은 `fold_split_masked.json` 로드로
고정되어 세션/계정이 달라져도 동일하게 유지됩니다 (데이터 소스나 제외 목록이 바뀌면 9번 셀 assert에서
중단되므로 조용히 어긋날 수 없음).

⏱ **시간 주의**: medium × 100 epochs × 5 folds는 Colab 세션 한도를 넘길 수 있습니다.
처음에는 `run_kfold_yolo(config, fold_yaml_paths, max_folds=1)`로 fold 1개 소요 시간을 가늠해 보세요.


In [ ]:
# [14. 5-fold 학습 + 리포팅 실행]
def run_kfold_yolo(config, fold_yaml_paths, max_folds=None):
    print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
    n_folds = max_folds if max_folds is not None else config['data']['n_splits']

    checkpoints, fold_metrics = [], []
    for fi in range(n_folds):
        dst = train_fold_yolo(
            fold_idx=fi,
            fold_yaml=fold_yaml_paths[fi],
            model_variant=config['model']['variant'],
            model_tag=config['model']['tag'],
            train_cfg=config['train'],
            local_output_dir=config['output']['local_output_dir'],
            backup_dir=config['output']['backup_dir'],
        )
        checkpoints.append(dst)
        if dst is None:
            print(f'[fold {fi}] 체크포인트 없음 — 리포팅 생략')
            continue

        metrics = report_fold_result_yolo(fi, dst, fold_yaml_paths[fi])
        fold_metrics.append(metrics)
        print(f"[fold {fi}] 완료 | mAP@0.5:0.95: {metrics['map']:.4f} | mAP@0.5: {metrics['map_50']:.4f}")

    print(f'\n▶ {n_folds}폴드 학습 완료')
    return {'checkpoints': checkpoints, 'fold_metrics': fold_metrics}

# 소요 시간 가늠용 sanity check: run_result = run_kfold_yolo(config, fold_yaml_paths, max_folds=1)
run_result = run_kfold_yolo(config, fold_yaml_paths)


In [ ]:
# [15. fold 결과 요약] Task2의 summarize_kfold_results/summarize_per_class와 같은 형태로 재구성합니다.
#  Ultralytics의 box.maps가 이미 class index(0~73)로 고정 정렬되어 있어, torchmetrics 기반 버전보다 단순합니다.
def summarize_kfold_results_yolo(fold_metrics, tag):
    """fold별 mAP 리스트를 받아 평균±표준편차를 출력합니다."""
    map_vals = [m['map'] for m in fold_metrics]
    map50_vals = [m['map_50'] for m in fold_metrics]
    map_mean, map_std = float(np.mean(map_vals)), float(np.std(map_vals))
    map50_mean, map50_std = float(np.mean(map50_vals)), float(np.std(map50_vals))
    print(f"\n{'='*50}\n{tag} 최종 결과 ({len(fold_metrics)}-fold 평균)\n"
          f"mAP@0.5:0.95: {map_mean:.4f} ± {map_std:.4f}\n"
          f"mAP@0.5: {map50_mean:.4f} ± {map50_std:.4f}\n{'='*50}")
    return {'map': (map_mean, map_std), 'map_50': (map50_mean, map50_std)}


def summarize_per_class_yolo(fold_metrics, label2cat, label_counts, valid_pivot):
    """클래스(라벨)별 mAP@0.5:0.95를 fold 평균으로 집계합니다.

    valid_pivot(라벨 x fold의 valid 박스 수)을 이용해, 그 fold의 valid에 해당 라벨 인스턴스가
    하나도 없었던 경우는 집계에서 제외합니다 (그 fold의 AP가 0으로 나와도 실제 성능이 아니라
    '평가 대상 없음'이기 때문).
    """
    rows = []
    for label in sorted(label2cat):
        aps = []
        for fi, m in enumerate(fold_metrics):
            if valid_pivot.loc[label, f'fold{fi}_valid'] == 0:
                continue
            aps.append(m['map_per_class'][label - 1])   # class index = label - 1
        rows.append({
            'label': label,
            'category_id': label2cat[label],
            'total_count': label_counts.get(label, 0),
            'mean_AP': round(float(np.mean(aps)), 4) if aps else -1,
            'std_AP': round(float(np.std(aps)), 4) if aps else 0,
            'valid_folds': len(aps),
        })
    return pd.DataFrame(rows).sort_values('mean_AP', ascending=False)


kfold_summary = summarize_kfold_results_yolo(run_result['fold_metrics'], config['model']['tag'])

label_counts = compute_label_counts(config['data']['dataset_dir'])
per_class_df = summarize_per_class_yolo(run_result['fold_metrics'], label2cat, label_counts, valid_pivot)
per_class_df   # mean_AP 내림차순 - RF-DETR(Task2) 결과와 클래스별로 대조해볼 수 있음


## 16~18. test 5-fold 앙상블 추론 → WBF 융합 → 클래스별 예측 시각화

test에는 GT가 없으므로 mAP 계산 없이: 5개 fold 모델의 예측을 전부 수집(합집합) → **WBF**로 이미지
단위 융합 → 클래스별 crop 시각화(confidence 표시)로 육안 점검합니다. YOLO 성능을 눈으로 확인하는
것이 목적이라 Kaggle 제출용 CSV는 만들지 않습니다.


In [ ]:
# [16. test 추론 - 5 fold 합집합 수집]
#  Task2의 collect_predictions_ensemble()과 동일한 반환 스키마로 로컬 정의합니다.
#  (rfdetr의 supervision.Detections 대신 ultralytics Results.boxes를 읽는 부분만 다름)
#  ⚠ YOLO의 cls는 0-indexed(0~73)라서, cat2label/label2cat 라벨 체계(1~74)에 맞추기 위해 +1 해서 저장합니다.
def collect_predictions_ensemble_yolo(checkpoints, image_dir, conf_thr=0.05,
                                      extensions=('.png', '.jpg', '.jpeg')):
    """test 폴더 전체에 대해 fold별 YOLO 체크포인트의 예측을 모아 이미지별로 병합합니다 (합집합)."""
    models = [get_yolo_model(checkpoint_path=p) for p in checkpoints]
    file_names = sorted(fn for fn in os.listdir(image_dir) if fn.lower().endswith(extensions))

    all_data = []
    for file_name in file_names:
        img_path = os.path.join(image_dir, file_name)
        image = np.array(Image.open(img_path).convert('RGB'))

        boxes_list, labels_list, scores_list, fold_list = [], [], [], []
        for fold_idx, model in enumerate(models):
            r = model.predict(img_path, conf=conf_thr, verbose=False)[0]
            n = len(r.boxes)
            if n == 0:
                continue
            boxes_list.append(r.boxes.xyxy.cpu().numpy())
            labels_list.append(r.boxes.cls.cpu().numpy().astype(int) + 1)   # 0-idx -> 라벨(1~74)
            scores_list.append(r.boxes.conf.cpu().numpy())
            fold_list.extend([fold_idx] * n)

        if boxes_list:
            pred_boxes = torch.tensor(np.concatenate(boxes_list), dtype=torch.float32)
            pred_labels = torch.tensor(np.concatenate(labels_list), dtype=torch.int64)
            pred_scores = torch.tensor(np.concatenate(scores_list), dtype=torch.float32)
        else:
            pred_boxes = torch.zeros((0, 4), dtype=torch.float32)
            pred_labels = torch.zeros((0,), dtype=torch.int64)
            pred_scores = torch.zeros((0,), dtype=torch.float32)

        all_data.append({
            'file_name': file_name, 'image': image,
            'pred_boxes': pred_boxes, 'pred_labels': pred_labels, 'pred_scores': pred_scores,
            'pred_fold': torch.tensor(fold_list, dtype=torch.int64),
        })

    del models
    torch.cuda.empty_cache()
    return all_data

TEST_IMG_DIR = os.path.join(DATA_ROOT, 'test_images')
ckpts = [p for p in run_result['checkpoints'] if p]
assert len(ckpts) == config['data']['n_splits'], f"체크포인트 누락: {run_result['checkpoints']}"

test_pred_data = collect_predictions_ensemble_yolo(ckpts, TEST_IMG_DIR, conf_thr=0.05)
print('test 이미지 수:', len(test_pred_data))
print('합집합 예측 박스 수:', sum(len(d['pred_boxes']) for d in test_pred_data))


In [ ]:
# [17. WBF(Weighted Box Fusion) 융합] Task2 Kaggle 노트북과 동일한 함수입니다 (예측 스키마가 같아 그대로 재사용).
#  합집합 그대로면 같은 알약이 fold 수만큼 중복되어 육안 확인 시에도 같은 알약이 여러 번 겹쳐 보입니다.
def fuse_predictions_wbf(pred_data, n_models, iou_thr=0.55, skip_box_thr=0.05):
    """collect_predictions_ensemble 계열 결과를 이미지 단위 WBF로 융합합니다. (로컬 정의)"""
    fused = []
    for d in pred_data:
        h, w = d['image'].shape[:2]
        scale = np.array([w, h, w, h], dtype=np.float32)

        if len(d['pred_boxes']) == 0:
            fused.append({'file_name': d['file_name'], 'image': d['image'],
                          'pred_boxes': torch.zeros((0, 4), dtype=torch.float32),
                          'pred_labels': torch.zeros((0,), dtype=torch.int64),
                          'pred_scores': torch.zeros((0,), dtype=torch.float32)})
            continue

        boxes_list, scores_list, labels_list = [], [], []
        for fi in range(n_models):
            m = (d['pred_fold'] == fi).numpy()
            b = d['pred_boxes'].numpy()[m] / scale
            boxes_list.append(np.clip(b, 0.0, 1.0).tolist())
            scores_list.append(d['pred_scores'].numpy()[m].tolist())
            labels_list.append(d['pred_labels'].numpy()[m].tolist())

        boxes, scores, labels = weighted_boxes_fusion(
            boxes_list, scores_list, labels_list,
            iou_thr=iou_thr, skip_box_thr=skip_box_thr)

        fused.append({
            'file_name': d['file_name'], 'image': d['image'],
            'pred_boxes': torch.tensor(np.asarray(boxes) * scale, dtype=torch.float32),
            'pred_labels': torch.tensor(labels, dtype=torch.int64),
            'pred_scores': torch.tensor(scores, dtype=torch.float32),
        })
    return fused

fused_data = fuse_predictions_wbf(test_pred_data, n_models=len(ckpts),
                                  iou_thr=WBF_IOU_THR, skip_box_thr=0.05)
n_before = sum(len(d['pred_boxes']) for d in test_pred_data)
n_after = sum(len(d['pred_boxes']) for d in fused_data)
print(f'융합 전 박스 {n_before}개 -> 융합 후 {n_after}개')


In [ ]:
# [18. 클래스별 예측 시각화] Task2 Kaggle 노트북과 동일한 함수입니다 (YOLO 성능 육안 확인이 목적 - 전체 예측 표시).
#  각 crop 제목: 1줄째 = 파일명(전체), 2줄째 = confidence score (클래스 내 score 내림차순 정렬)
#  ⚠ 한글 폰트 미지원 환경(Colab 기본 matplotlib) 대비, plot에 렌더링되는 텍스트는 영어로만 표기합니다.
def show_pred_class_crops(fused_data, label2cat, score_thr=0.3, ncols=6, pad=8, classes=None):
    """WBF 융합 예측을 클래스별 crop grid로 '전부' 표시합니다."""
    by_label = defaultdict(list)
    for d in fused_data:
        h, w = d['image'].shape[:2]
        keep = d['pred_scores'] >= score_thr
        for box, label, score in zip(d['pred_boxes'][keep], d['pred_labels'][keep],
                                     d['pred_scores'][keep]):
            x1, y1, x2, y2 = box.tolist()
            x1, y1 = max(0, int(x1) - pad), max(0, int(y1) - pad)
            x2, y2 = min(w, int(x2) + pad), min(h, int(y2) + pad)
            if x2 <= x1 or y2 <= y1:
                continue
            by_label[int(label)].append((d['image'][y1:y2, x1:x2], float(score), d['file_name']))

    print(f'score >= {score_thr} 기준, 예측이 존재하는 클래스: {len(by_label)}개')
    targets = sorted(by_label) if classes is None else [l for l in classes if l in by_label]
    for label in targets:
        items = sorted(by_label[label], key=lambda t: -t[1])   # score 내림차순, 전부 표시

        nrows = math.ceil(len(items) / ncols)
        fig, axes = plt.subplots(nrows, ncols, figsize=(2.2 * ncols, 2.8 * nrows))
        axes = np.atleast_1d(axes).reshape(-1)
        for ax in axes:
            ax.axis('off')
        for ax, (crop, score, fn) in zip(axes, items):
            ax.imshow(crop)
            ax.set_title(f'{fn}\nscore={score:.2f}', fontsize=5)
        fig.suptitle(f'label={label} / category_id={label2cat[label]}'
                     f'  (score>={score_thr}: {len(items)} preds)', fontsize=10)
        plt.tight_layout(rect=[0, 0, 1, 0.97])
        plt.show()

# 특정 클래스만 보려면 classes=[57, 58] 처럼 '라벨' 번호로 지정하세요 (57~74 = test 전용 18종).
show_pred_class_crops(fused_data, label2cat, score_thr=VIS_SCORE_THR)


## 산출물

- `{backup_dir}/{tag}_fold{0..4}_best.pt` — 5-fold YOLOv8 체크포인트 (cls gain 1.5로 분류 강화 학습)
  (다음 단계에서 task2-masked RF-DETR와 fold-matched WBF 앙상블에 사용 — 두 실험이 같은
  `fold_split_masked.json`으로 fold를 나눴으므로 `fold{i}`끼리 valid 이미지 집합이 동일함)
- `{backup_dir}/{tag}_fold{i}_results.csv` / `_results.png` — fold별 학습 곡선 (Ultralytics 자동 생성)
- 이 노트북은 체크포인트 생성 + 육안 QA가 목적이라 Kaggle 제출용 CSV는 만들지 않습니다.
  최종 제출은 task2-masked(RF-DETR) + task4-masked(YOLOv8) 체크포인트를 함께 불러와 fold-matched WBF로
  앙상블하는 별도 노트북에서 진행합니다.
